## Data Ingestion for Deep RAG

In this notebook, we'll load extracted data into Qdrant vector database:

- **Markdown**: Page-level chunks with metadata
- **Tables**: Separate documents with context and page numbers
- **Images**: Text descriptions embedded (generated in notebook 06-01b)
- **Hybrid Search**: Dense (semantic) + Sparse (keyword) embeddings

**Prerequisites:**
- Run notebook 06-01 first to extract PDFs
- Run notebook 06-01b to generate image descriptions
- Qdrant server running on localhost:6333
- Google API key set in .env file

**Output:**
- Single Qdrant collection with all content types
- Rich metadata for filtering (company, year, quarter, doc_type, page)
- Deduplication using file hashes

**Make Sure You Have Your QDRANT Vector DB Docker Running**

https://qdrant.tech/

| Point            | **Qdrant** | **Chroma**       | **FAISS** | Weaviate     | Milvus | Pinecone |
| ---------------- | ---------- | ---------------- | --------- | ------------ | ------ | -------- |
| Open Source      | ✅ Yes      | ✅ Yes            | ✅ Yes     | ⚠️ Open-core | ✅ Yes  | ❌ No     |
| DB vs Library    | DB         | DB (dev-focused) | Library   | DB           | DB     | Managed  |
| Hybrid Search    | ✅ Native   | ❌                | ❌         | ✅            | ⚠️     | ✅        |
| Metadata Filter  | ✅ Strong   | ⚠️ Basic         | ❌         | ✅            | ✅      | ✅        |
| Production Ready | ✅ Yes      | ❌ (POC)          | ❌         | ✅            | ✅      | ✅        |
| Local / Offline  | ✅ Yes      | ✅ Yes            | ⚠️        | ⚠️           | ⚠️     | ❌        |


jieba分词测试

### 0. Qdrant API Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from qdrant_client import QdrantClient

qdrant_client = QdrantClient(
    url="https://cb36196c-4fa0-4a84-af87-6a84b091f2b3.us-west-1-0.aws.cloud.qdrant.io:6333",
    api_key=os.getenv("QDRANT_API_KEY"),
)

print(qdrant_client.get_collections())

collections=[CollectionDescription(name='industrial_docs')]


In [ ]:
# qdrant_client = QdrantClient(
#     url="http://localhost:6333"
# )

# print(qdrant_client.get_collections())

collections=[CollectionDescription(name='financial_docs')]


### 1. Setup and Imports

In [2]:
import hashlib
from pathlib import Path

from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_qdrant import QdrantVectorStore, RetrievalMode, FastEmbedSparse

from langchain_core.documents import Document
from qdrant_client import QdrantClient

### 2. Configuration

In [3]:
# Paths
MARKDOWN_DIR = "industrial_data/markdown"
TABLES_DIR = "industrial_data/tables"
IMAGES_DESC_DIR = "industrial_data/images_desc"

# Qdrant Configuration
COLLECTION_NAME = "industrial_docs"
EMBEDDING_MODEL = "models/gemini-embedding-001"

### 3. Initialize Embeddings and Client

In [4]:
# Embeddings
embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)


In [5]:
import os

# 在导入任何网络库之前清除代理设置
for var in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "http_proxy",
    "https_proxy",
    "ALL_PROXY",
    "all_proxy",
]:
    os.environ.pop(var, None)

# 强制设置不代理本地地址
os.environ["NO_PROXY"] = "localhost,127.0.0.1"


In [6]:
result = embeddings.embed_query('anything')
len(result)

3072

In [8]:
import jieba
import re
s = """这是一条测试文本"""
seg = jieba.cut(s, cut_all=False)
result = (' ').join([token for token in seg if token.strip() and not re.match(r"^[\s\W]+$", token)])

print(result)

这是 一条 测试 文本


In [9]:
# 重写FastEmbedSparse类，在外面套一层分词器
from typing import List
class JiebaFastEmbedSparse(FastEmbedSparse):
    """在 FastEmbedSparse 外层包一个 jieba 分词预处理"""

    def _preprocess(self, text: str) -> str:
        seg = jieba.cut(text, cut_all=False)
        result = (' ').join([token for token in seg if token.strip() and not re.match(r"^[\s\W]+$", token)])
        return result

    def embed_documents(self, texts: List[str]) -> List:
        tokenized = [self._preprocess(t) for t in texts]
        return super().embed_documents(tokenized)

    def embed_query(self, text: str) -> List:
        tokenized = self._preprocess(text)
        return super().embed_query(tokenized)

In [10]:
import os

os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7897"
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7897"

In [11]:
sparse_embeddings = JiebaFastEmbedSparse(model_name="Qdrant/bm25")
sparse_embeddings_old = FastEmbedSparse(model_name="Qdrant/bm25")
print(sparse_embeddings.embed_query("""这是一条测试文本"""))
print(sparse_embeddings_old.embed_query("""这是一条测试文本"""))

indices=[25843656, 607096161, 875482371, 2116377680] values=[1.0, 1.0, 1.0, 1.0]
indices=[344787478] values=[1.0]


In [12]:
result = sparse_embeddings.embed_query('hi hello')
result

SparseVector(indices=[948991206, 613153351], values=[1.0, 1.0])

In [13]:
result = sparse_embeddings.embed_documents(['hi', 'hello'])
result

[SparseVector(indices=[948991206], values=[1.6877434821696136]),
 SparseVector(indices=[613153351], values=[1.6877434821696136])]

### 4. Create or Recreate Collection

In [44]:
# # Create vector store at Remote location
vector_store = QdrantVectorStore.from_documents(
    documents=[],
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    url="https://cb36196c-4fa0-4a84-af87-6a84b091f2b3.us-west-1-0.aws.cloud.qdrant.io:6333",
    api_key=os.getenv("QDRANT_API_KEY"),
    collection_name=COLLECTION_NAME,
    retrieval_mode=RetrievalMode.HYBRID,
    force_recreate=False,
)

In [ ]:
# Create vector store at local computer
# vector_store = QdrantVectorStore.from_documents(
#     documents=[],
#     embedding=embeddings,
#     sparse_embedding=sparse_embeddings,
#     url="http://localhost:6333", 
#     collection_name = COLLECTION_NAME,
#     retrieval_mode=RetrievalMode.HYBRID,
#     force_recreate=False
# )

In [45]:
vector_store.client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='industrial_docs')])

### 5. Helper Functions

In [17]:
def extract_metadata_from_filename(filename: str):
    """
    Extract metadata from filename.

    Expected format: Category_Index_Technology.pdf
    Examples:
        - Industrial_Automation_Safety_Part01_General.pdf
        - Industrial_Automation_Safety_Part02_Pressure_Transmitter.pdf
    """

    filename = filename.replace(".pdf", "").replace(".md", "")
    parts = filename.split("_")

    return {
        "category": "_".join(parts[:3]),
        "index": parts[3],
        "technology": "_".join(parts[4:]),
    }


extract_metadata_from_filename("Industrial_Automation_Safety_Part01_General.pdf")

{'category': 'Industrial_Automation_Safety',
 'index': 'Part01',
 'technology': 'General'}

In [25]:
# def extract_metadata_from_filename(filename: str):
#     """
#     Extract metadata from filename.
    
#     Expected format: CompanyName DocType [Quarter] Year.pdf
#     Examples:
#         - Amazon 10-Q Q1 2024.pdf
#         - Microsoft 10-K 2023.pdf
#     """

#     filename = filename.replace('.pdf', '').replace('.md', '')
#     parts = filename.split()

#     return {
#         'company_name': parts[0],
#         'doc_type': parts[1],
#         'fiscal_quarter': parts[2] if len(parts)==4 else None,
#         'fiscal_year': parts[-1]
#     }

# extract_metadata_from_filename('apple 10-k 2023.md')

In [39]:
def compute_file_hash(file_path: Path):

    sha256_hash = hashlib.sha256()

    with open(file_path, 'rb') as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)

    return sha256_hash.hexdigest()


In [40]:
compute_file_hash(
    Path(r"industrial_data\markdown\Industrial_Automation_Safety_Part01_General.md")
)

'1686d6c47aa7a5a090919c317a41df531ab7323940a942518791b7e96bd30d03'

In [ ]:
compute_file_hash(Path(r'data\rag-data\markdown\amazon\amazon 10-k 2023 copy.md'))

In [46]:
# get the list of ingested file
all_points = vector_store.client.scroll(
    collection_name=COLLECTION_NAME,
    limit=10_00,
    with_payload=True,
    offset=None
)

In [47]:
all_points[1]


In [48]:
# if it is first time, you will get None
# all_points[0][0].payload['metadata']['file_hash']
if all_points and len(all_points[0]) > 0:
    file_hash = all_points[0][0].payload.get("metadata", {}).get("file_hash")
else:
    file_hash = None
    print("No point exist yet")
 
print(file_hash)

5b16ace1392e652ef8e144e4ebdef34e6724b61cb9931a883205fa1cde2820b6


In [49]:
def get_processed_hashes():
    
    processed_hashes = set()
    offset = None

    while True:
        points, offset = vector_store.client.scroll(
                            collection_name=COLLECTION_NAME,
                            limit=10_000,
                            with_payload=True,
                            offset=offset
                        )

        if not points:
            break
        
        processed_hashes.update(
            point.payload.get("metadata", {}).get("file_hash")
            for point in points
            if point.payload.get("metadata", {}).get("file_hash") is not None
        )

        if offset is None:
            break

    return processed_hashes

In [50]:
processed_hashes = get_processed_hashes()

In [51]:
len(processed_hashes)

183

In [26]:
# extract the page number from the file path
import re

def extract_page_number(file_path: Path):
    pattern = r'page_(\d+)'
    match = re.search(pattern=pattern, string=file_path.stem)
    return int(match.group(1)) if match else None

In [27]:
file_path = Path(
    r"industrial_data\tables\Industrial_Automation_Safety_Part01_General\table_1_page_13.md"
)
extract_page_number(file_path)

13

### 6. Ingestion Function

In [32]:
file_path = Path(
    r"industrial_data\images_desc\Industrial_Automation_Safety_Part02_Pressure_Transmitter\page_17.md"
)
path_str = str(file_path)
if 'markdown' in path_str:
    content_type = 'text'
    doc_name = file_path.name
    img_path = None
elif 'tables' in path_str:
    content_type = 'tables'
    doc_name = file_path.parent.name
    img_path = None
elif 'images_desc' in path_str:
    content_type = 'image'
    doc_name = file_path.parent.name
    img_path = file_path
else:
    content_type = 'unknown'
    doc_name = file_path.name
    img_path = None
print(content_type, doc_name, img_path)

image Industrial_Automation_Safety_Part02_Pressure_Transmitter industrial_data\images_desc\Industrial_Automation_Safety_Part02_Pressure_Transmitter\page_17.md


In [33]:
def ingest_file_in_db(file_path, processed_hashes):

    file_hash = compute_file_hash(file_path)
    if file_hash in processed_hashes:
        print(f"Following file has been already uploaded: {file_path}")

    path_str = str(file_path)
    if 'markdown' in path_str:
        content_type = 'text'
        doc_name = file_path.name
        img_path = None
    elif 'tables' in path_str:
        content_type = 'tables'
        doc_name = file_path.parent.name
        img_path = None
    elif 'images_desc' in path_str:
        content_type = 'image'
        doc_name = file_path.parent.name
        img_path = file_path
    else:
        content_type = 'unknown'
        doc_name = file_path.name
        img_path = None

    content = file_path.read_text(encoding='utf-8')

    base_metadata = extract_metadata_from_filename(doc_name)

    base_metadata.update({
        'content_type': content_type,
        'file_hash': file_hash,
        'source_file': doc_name,
        'image_path':img_path
    })

    if content_type == 'text':
        # write method for ingesting markdown data
        pages = content.split('<!-- page break -->')
        documents = []
        for idx, page in enumerate(pages, start=1):
            metadata = base_metadata.copy()
            metadata.update({'page': idx})
            documents.append(Document(page_content=page, metadata=metadata))

        vector_store.add_documents(documents)

    else:
        # write method to ingest images desc and tables .md data
        page_num = extract_page_number(file_path)
        metadata = base_metadata.copy()
        metadata.update({'page': page_num})
        documents = [Document(page_content=content, metadata=metadata)]

        vector_store.add_documents(documents)


    processed_hashes.add(file_hash)


In [ ]:
file_path = Path(r'data\rag-data\markdown\amazon\amazon 10-k 2023.md')
processed_hashes = get_processed_hashes()

if processed_hashes is None:
    processed_hashes = set()

ingest_file_in_db(file_path, processed_hashes)

In [ ]:
from tqdm import tqdm

base_path = Path("industrial_data")
all_md_files = list(base_path.rglob("*.md"))

processed_hashes = get_processed_hashes()

if processed_hashes is None:
    processed_hashes = set()

for md_file in tqdm(all_md_files):
    ingest_file_in_db(md_file, processed_hashes)

### 8. Verify Ingestion

In [52]:
collection_info = vector_store.client.get_collection(COLLECTION_NAME)
collection_info

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=630, points_count=630, segments_count=2, config=CollectionConfig(params=CollectionParams(vectors={'': VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors={'langchain-sparse': SparseVectorParams(index=None, modifier=None)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush

### 9. Test Search

In [55]:
query = "控制阀的安全要求"
results = vector_store.similarity_search(query)

In [56]:
results

[Document(metadata={'category': 'Industrial_Automation_Safety', 'index': 'Part04', 'technology': 'Control_Valve', 'content_type': 'text', 'file_hash': '009b1cc41beb0046b02cdf5c9ca50af42fd5989d6cebf78d3ca1af3f8f0d64ec', 'source_file': 'Industrial_Automation_Safety_Part04_Control_Valve.md', 'image_path': None, 'page': 18, '_id': '25dbea09-3f5a-4912-84d5-0e11138ddce5', '_collection_name': 'industrial_docs'}, page_content='\n\n## 5\n\n## . 4 . 3 控制阀安装\n\n文件应当包括安装和特定的交付使用的说明 , 以及如果对安全是必要的话 , 还应当包括在控制阀安\n\n装和交付使用过程中可能发生的危险的警告 。 如 :\n\n- a)\n- 装配 、 定位和安装要求 , 如气动控制阀排气口可能对操作人员的眼睛造成潜在伤害 , 应给出合 理的位置和朝向的建议 ;\n- b) 如有必要 , 给出保护接地说明 ;\n- c) 与电源的连接 ;\n- d) 电源布线要求 ;\n- e)\n- 如有必要 , 给出任何外部开关或断路器 ( 见 6. 1 1 . 2. 1) 和外部过流保护装置 ( 见 9. 5. 1) 的要求 , 以 及将这些开关或电路断路器设置在控制阀近旁的建议 ;\n- f ) 特殊维护要求 , 如空气质量 。\n\n通过目视检查来检验是否合格 。\n\n## 5 . 4 . 4 控制阀的操作\n\n使用说明应当包括 :\n\n- a) 操作控制件及其用于各种操作方式的标识 ;\n2. b)\n3. 不要将控制阀放在难以操作控制件的位置的说明 ;\n- c) 与附件和其他设备互连的说明 , 包括指出适用的附件 、 可拆卸的零部件和任何专用的材料\n- d) 如有必要 , 给出间歇工作限值的规范 ;\n- e) 在控制阀上使用的与安